# Simulation Code T = 2

## Import libraries

In [1]:
#import torch
import pandas as pd
import time
from utils.dynamicRieszFunctions_org import *
from tqdm import tqdm

## Estimation Settings

In [2]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 1,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

rf_f_settings = {
    'poly_degree' : 1,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

#### Define treatment probability function

In [3]:
# Bounds (only for truncated distributions)
lower = 0.3
upper = 0.7

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Define func_link function (nonlinear probability function from Bradic)
def func_link(x):
    return (x > 0) * torch.abs(x) / (torch.abs(x) + 1) + (x < 0) / (torch.abs(x) + 1)

# Define a truncated func_link function
def truncated_link(x):
    return lower + (upper - lower) * func_link(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * ((x < - 1/2) + (x < 1/2))

In [4]:
treatment_probability_func = truncated_logistic

#### True values: simulate ATE for experimental dataset

In [5]:
torch.manual_seed(123);

# Parameters
N = 500000
tmax = 1

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX = 3
dimS = 3

prob_G = 0.5 * torch.ones(N * tmax, 1)
G_all = torch.bernoulli(prob_G).reshape(-1,1).float()

X_all = torch.randn(N * tmax, dimX, dtype=torch.float64)

prob_D = X_all @ torch.tensor([-1, 1, -1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1) 
# prob_D = 0.5 * torch.ones(N * tmax, 1)
D_all = torch.bernoulli(treatment_probability_func(prob_D)).reshape(-1,1)

S0_all = X_all + torch.randn(N * tmax, dimS) * 0.5
S1_all = S0_all + 1 * 1 * (1 + torch.randn(N * tmax, dimS))
# S_all = (1 - G_all) * D_all * S1_all + (1 - (1 - G_all) * D_all) * S0_all


Y_eps = torch.randn(N * tmax, 1) * 1
# Y0_all = (S0_all @ torch.tensor([-1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
#       + X_all @ torch.tensor([1, 1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
#       + Y_eps)

# Y1_all = (S1_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1) 
#       + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float32).reshape(-1,1)
#       + Y_eps)

Y0_all =  (S0_all @ torch.tensor([1, 1, -1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1)
      + Y_eps)

Y1_all =  (S1_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1)
      + Y_eps)


In [6]:
true_value = torch.mean(Y1_all[((1 - G_all)).squeeze().bool(),:]) - torch.mean(Y0_all[((1 - G_all)).squeeze().bool(),:])
true_value

tensor(1.0027, dtype=torch.float64)

#### Simulation settings:

In [7]:
seed = 0
torch.manual_seed(seed);

# Parameters
N = 2000
tmax = 40
 
# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX = 3
dimS = 3

#### Generate data

In [8]:
prob_G = 0.5 * torch.ones(N * tmax, 1)
G_all = torch.bernoulli(prob_G).reshape(-1,1)

X_all = torch.randn(N * tmax, dimX, dtype=torch.float64)

prob_D = X_all @ torch.tensor([-1, 1, -1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1) 
D_all = torch.bernoulli(treatment_probability_func(prob_D)).reshape(-1,1)

S0_all = X_all + torch.randn(N * tmax, dimS) * 0.5
S1_all = S0_all + 1 * 1 * (1 + torch.randn(N * tmax, dimS))
S_all = (1 - G_all) * D_all * S1_all + (1 - (1 - G_all) * D_all) * S0_all


Y_eps = torch.randn(N * tmax, 1) * 1

Y_all = G_all * (S_all @ torch.tensor([1,1, -1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1) 
      + X_all @ torch.tensor([1, 1, 1] + [0] * (dimX - 3), dtype=torch.float64).reshape(-1,1)
      + Y_eps)


## Estimation:

#### Estimation settings

In [9]:
folds = 5
time0 = time.time()

In [12]:
D_all.shape

torch.Size([80000, 1])

In [14]:
t = 0
# Get data for current iteration
G, X = G_all[(t) * N : (t+1) * N, :], X_all[(t) * N : (t+1) * N, :]
D, S = D_all[(t) * N : (t+1) * N], S_all[(t) * N : (t+1) * N]
Y = Y_all[(t) * N : (t+1) * N]

In [21]:
print(Y.shape)
print(G.shape) 
print(D.shape)
print(X.shape)
print(S.shape)

torch.Size([2000, 1])
torch.Size([2000, 1])
torch.Size([2000, 1])
torch.Size([2000, 3])
torch.Size([2000, 3])


#### Estimation 

In [10]:
pred_theta = torch.zeros(tmax,5)
pred_theta_subset= torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)
pred_sig_subset= torch.zeros(tmax,5)
for t in tqdm(range(0,tmax)):

    # Get data for current iteration
    G, X = G_all[(t) * N : (t+1) * N, :], X_all[(t) * N : (t+1) * N, :]
    D, S = D_all[(t) * N : (t+1) * N], S_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]
    D_changed = D.clone()
    Y_changed = Y.clone()
    ###################
    D_changed[G.bool()] = 0
    Y_changed[(1-G).bool()] =  0
   
    
    pred_theta[t,1], pred_sig[t,1]= estimateDynamicRiesz(Y_changed, G, X, D_changed, S, folds,
                                                                     method_a = "LASSO", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "LASSO", lasso_f_settings = lasso_f_settings, seed= t)
    pred_theta[t,2], pred_sig[t,2]= estimateDynamicRiesz(Y_changed, G, X, D_changed, S, folds,
                                                                     method_a = "RF", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "RF", lasso_f_settings = lasso_f_settings, seed= t)
    pred_theta[t,3], pred_sig[t,3]= estimateDynamicRiesz(Y_changed, G, X, D_changed, S, folds,
                                                                     method_a = "Net", lasso_a_settings = lasso_a_settings,
                                                                        method_f = "Net", lasso_f_settings = lasso_f_settings, seed= t)


                                                                     





100%|██████████| 40/40 [1:02:56<00:00, 94.41s/it]


In [16]:
pred_theta.mean(dim = 0)

tensor([0.0000, 1.0630, 1.0235, 0.9859, 0.0000])

In [18]:
pred_sig.mean(dim = 0)

tensor([ 0.0000,  6.1981,  5.3887, 21.3291,  0.0000])

In [13]:
true_value

tensor(1.0027, dtype=torch.float64)

## Compute statistics

#### Get true value

#### Compute statistics

In [14]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["" , "LASSO", "RF", "Net", ""],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

In [15]:
print(data)

{'Method': ['', 'LASSO', 'RF', 'Net', ''], 'Bias': tensor([-1.0027,  0.0603,  0.0208, -0.0168, -1.0027]), 'RMSE': tensor([1.0027, 0.1364, 0.1672, 0.4792, 1.0027]), 'Coverage': tensor([0.0000, 0.9250, 0.7750, 1.0000, 0.0000]), 'Interval Length': tensor([0.0000, 0.5433, 0.4723, 1.8696, 0.0000])}
